# 📘 Notebook 01: Data Ingestion and Chunking

## 🔍 Introduction

This notebook implements the **data ingestion and preprocessing pipeline** for the Agentic Retrieval-Augmented Generation (RAG) system developed for Al-Farabi Kazakh National University (KazNU).

The primary objective is to collect, clean, and structure unstructured data from multiple institutional sources, including:

- Official university PDF documents (e.g., academic policies, regulations)
- Web pages from the KazNU website

Since no official API is available, the system relies on **document loading and web data extraction** to build a comprehensive knowledge base.

The pipeline performs the following key steps:

1. **Document Loading** – Import PDFs and web content  
2. **Text Cleaning** – Normalize and preprocess raw text  
3. **Chunking** – Split documents into smaller, overlapping segments for efficient retrieval  
4. **Metadata Enrichment** – Attach source and contextual information  
5. **Data Analysis** – Evaluate chunk distribution and quality  
6. **Persistence** – Save processed chunks for downstream vector database creation  

The output of this notebook serves as the **foundation for embedding generation and vector store construction** in the next stage of the RAG pipeline.

In [ ]:
# Import required libraries
import os
import pickle
from pathlib import Path
from typing import List, Dict, Any
import pandas as pd
from tqdm import tqdm
import re

# LangChain imports for document loading
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# For better PDF parsing (especially for tables)
import pdfplumber

# Configure paths
PROJECT_ROOT = Path.cwd().parent  # Go up one level from notebooks/
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PDF_DIR = DATA_RAW_DIR / "pdfs"

# Create directories if they don't exist
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {PROJECT_ROOT}")
print(f"PDF Directory: {PDF_DIR}")
print(f"Processed Data Directory: {DATA_PROCESSED_DIR}")

# %% [markdown]
# ## 2. Load PDF Documents

# %%
def load_pdfs_from_directory(pdf_dir: Path) -> List[Document]:
    """
    Load all PDF files from the specified directory using PyPDFLoader.
    """
    documents = []
    pdf_files = list(pdf_dir.glob("*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files:")
    for pdf_file in pdf_files:
        print(f"  - {pdf_file.name}")
    
    for pdf_path in tqdm(pdf_files, desc="Loading PDFs"):
        try:
            loader = PyPDFLoader(str(pdf_path))
            pdf_docs = loader.load()
            
            # Add source metadata to each document
            for doc in pdf_docs:
                doc.metadata["source"] = pdf_path.name
                doc.metadata["source_type"] = "pdf"
                doc.metadata["file_path"] = str(pdf_path)
            
            documents.extend(pdf_docs)
            print(f"  ✓ Loaded {len(pdf_docs)} pages from {pdf_path.name}")
        except Exception as e:
            print(f"  ✗ Error loading {pdf_path.name}: {e}")
    
    return documents

# Load PDFs
pdf_documents = load_pdfs_from_directory(PDF_DIR)
print(f"\nTotal PDF documents loaded: {len(pdf_documents)} pages")

# %% [markdown]
# ## 3. Load Web URLs

# %%
def load_urls_from_file(url_file: Path) -> List[str]:
    """
    Read URLs from a text file (one URL per line).
    """
    with open(url_file, 'r', encoding='utf-8') as f:
        urls = [line.strip() for line in f if line.strip() and not line.startswith('#')]
    return urls

def load_web_documents(urls: List[str]) -> List[Document]:
    """
    Load web pages from URLs using WebBaseLoader.
    """
    documents = []
    
    print(f"\nLoading {len(urls)} URLs:")
    for i, url in enumerate(tqdm(urls, desc="Loading web pages")):
        try:
            loader = WebBaseLoader(url)
            web_docs = loader.load()
            
            # Add source metadata to each document
            for doc in web_docs:
                doc.metadata["source"] = url
                doc.metadata["source_type"] = "web"
                doc.metadata["url"] = url
            
            documents.extend(web_docs)
            print(f"  ✓ Loaded {url[:60]}...")
        except Exception as e:
            print(f"  ✗ Error loading {url}: {e}")
    
    return documents

# Load URLs from file
url_file = DATA_RAW_DIR / "urls.txt"
if url_file.exists():
    urls = load_urls_from_file(url_file)
    print(f"Found {len(urls)} URLs in urls.txt:")
    for url in urls[:5]:  # Show first 5
        print(f"  - {url}")
    if len(urls) > 5:
        print(f"  ... and {len(urls) - 5} more")
    
    # Load web documents
    web_documents = load_web_documents(urls)
    print(f"\nTotal web documents loaded: {len(web_documents)} pages")
else:
    print(f"Warning: {url_file} not found. Skipping web document loading.")
    web_documents = []

# %% [markdown]
# ## 4. Combine All Documents

# %%
# Combine PDF and web documents
all_documents = pdf_documents + web_documents
print(f"\nTotal combined documents: {len(all_documents)}")

# Display sample of first document
if all_documents:
    print("\n" + "="*80)
    print("SAMPLE DOCUMENT (first page):")
    print("="*80)
    print(f"Metadata: {all_documents[0].metadata}")
    print("\nContent Preview (first 500 chars):")
    print("-"*40)
    print(all_documents[0].page_content[:500])
    print("-"*40)

# %% [markdown]
# ## 5. Clean Text Content

# %%
def clean_text(text: str) -> str:
    """
    Clean and normalize text content.
    """
    if not text:
        return ""
    
    # Remove multiple newlines
    text = re.sub(r'\n\s*\n', '\n\n', text)
    
    # Remove multiple spaces
    text = re.sub(r' +', ' ', text)
    
    # Remove special characters but keep basic punctuation
    text = re.sub(r'[^\w\s\.\,\-\:\;\(\)\[\]\{\}\/\@]', '', text)
    
    # Strip leading/trailing whitespace
    text = text.strip()
    
    return text

# Clean all documents
print("Cleaning document text...")
for doc in tqdm(all_documents, desc="Cleaning"):
    doc.page_content = clean_text(doc.page_content)

print("✓ Text cleaning completed")

# %% [markdown]
# ## 6. Chunk Documents

# %%
def create_chunks(
    documents: List[Document],
    chunk_size: int = 1000,
    chunk_overlap: int = 200
) -> List[Document]:
    """
    Split documents into overlapping chunks for better retrieval.
    """
    # Initialize text splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""],
        keep_separator=True
    )
    
    # Split all documents
    print(f"Splitting {len(documents)} documents into chunks...")
    chunks = text_splitter.split_documents(documents)
    
    # Add chunk metadata
    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i
        chunk.metadata["chunk_size"] = len(chunk.page_content)
    
    return chunks

# Create chunks
chunk_size = 1000  # Adjust based on your needs
chunk_overlap = 200

chunks = create_chunks(
    all_documents,
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

print(f"\nCreated {len(chunks)} total chunks")
print(f"Average chunk size: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} characters")

# Display sample chunk
if chunks:
    print("\n" + "="*80)
    print("SAMPLE CHUNK:")
    print("="*80)
    print(f"Metadata: {chunks[0].metadata}")
    print("\nContent Preview (first 300 chars):")
    print("-"*40)
    print(chunks[0].page_content[:300])
    print("-"*40)

# %% [markdown]
# ## 7. Analyze Document Distribution

# %%
# Create a DataFrame for analysis
chunk_data = []
for chunk in chunks:
    chunk_data.append({
        "chunk_id": chunk.metadata.get("chunk_id"),
        "source": chunk.metadata.get("source", "unknown"),
        "source_type": chunk.metadata.get("source_type", "unknown"),
        "chunk_size": len(chunk.page_content)
    })

df_chunks = pd.DataFrame(chunk_data)

# Display statistics
print("CHUNK STATISTICS:")
print("-"*40)
print(f"Total chunks: {len(df_chunks)}")
print(f"By source type:")
print(df_chunks['source_type'].value_counts())
print(f"\nBy source file:")
print(df_chunks['source'].value_counts().head(10))

# Display size distribution
print(f"\nChunk size statistics:")
print(df_chunks['chunk_size'].describe())

# %% [markdown]
# ## 8. Save Processed Chunks

# %%
def save_chunks(chunks: List[Document], filename: str):
    """
    Save chunks to a pickle file for later use.
    """
    filepath = DATA_PROCESSED_DIR / filename
    
    # Convert chunks to serializable format
    chunks_data = []
    for chunk in chunks:
        chunks_data.append({
            "page_content": chunk.page_content,
            "metadata": chunk.metadata
        })
    
    with open(filepath, 'wb') as f:
        pickle.dump(chunks_data, f)
    
    print(f"✓ Saved {len(chunks)} chunks to {filepath}")
    return filepath

def save_chunks_as_json(chunks: List[Document], filename: str):
    """
    Save chunks as JSON for easy inspection.
    """
    filepath = DATA_PROCESSED_DIR / filename
    
    chunks_data = []
    for chunk in chunks:
        chunks_data.append({
            "content": chunk.page_content[:200] + "..." if len(chunk.page_content) > 200 else chunk.page_content,
            "metadata": chunk.metadata
        })
    
    df = pd.DataFrame(chunks_data)
    df.to_json(filepath, orient='records', indent=2)
    print(f"✓ Saved preview JSON to {filepath}")

# Save chunks in both formats
print("\nSAVING PROCESSED CHUNKS:")
print("-"*40)

# Save full chunks for vector store
pkl_file = save_chunks(chunks, "all_chunks.pkl")

# Save preview JSON for inspection
save_chunks_as_json(chunks, "chunks_preview.json")

# %% [markdown]
# ## 9. Summary Statistics

# %%
print("\n" + "="*80)
print("DATA INGESTION SUMMARY")
print("="*80)
print(f"PDF Documents: {len(pdf_documents)} pages")
print(f"Web Documents: {len(web_documents)} pages")
print(f"Total Documents: {len(all_documents)} pages")
print(f"Total Chunks: {len(chunks)}")
print(f"Average Chunk Size: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} chars")
print(f"Output Files:")
print(f"  - {DATA_PROCESSED_DIR / 'all_chunks.pkl'}")
print(f"  - {DATA_PROCESSED_DIR / 'chunks_preview.json'}")
print("="*80)

# %% [markdown]
# ## Next Steps
# 
# You're now ready to move to **Notebook 2: `02_vector_store_creation.ipynb`**
# 
# In the next notebook, you will:
# 1. Load these chunks
# 2. Generate embeddings using `all-MiniLM-L6-v2`
# 3. Create and persist a ChromaDB vector store

Project Root: d:\agentic-rag-dissertation\agentic-rag-university-assistant
PDF Directory: d:\agentic-rag-dissertation\agentic-rag-university-assistant\data\raw\pdfs
Processed Data Directory: d:\agentic-rag-dissertation\agentic-rag-university-assistant\data\processed
Found 3 PDF files:
  - Academic-policy_Академическая политика 2025-2026 (англ).pdf
  - AI_Regulation_2024.pdf
  - Booklet_KazNU_2025.pdf


Loading PDFs:  33%|███▎      | 1/3 [00:03<00:06,  3.50s/it]

  ✓ Loaded 33 pages from Academic-policy_Академическая политика 2025-2026 (англ).pdf


Loading PDFs:  67%|██████▋   | 2/3 [00:04<00:02,  2.13s/it]

  ✓ Loaded 14 pages from AI_Regulation_2024.pdf


...

  ✓ Loaded https://farabi.university/students/35?lang=en...


Loading web pages: 100%|██████████| 15/15 [01:04<00:00,  4.33s/it]


  ✓ Loaded https://farabi.university/students/keremet/18?lang=en...

Total web documents loaded: 15 pages

Total combined documents: 98

SAMPLE DOCUMENT (first page):
Metadata: {'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-05T14:42:26+05:00', 'author': 'ˆ@C3>2KE ˘;LO', 'moddate': '2025-11-05T14:42:26+05:00', 'title': 'Microsoft Word - \x10:045< ?>;8B8:0 0=3; 2025 -2026 0=3; (1) final', 'source': 'Academic-policy_Академическая политика 2025-2026 (англ).pdf', 'total_pages': 33, 'page': 0, 'page_label': '1', 'source_type': 'pdf', 'file_path': 'd:\\agentic-rag-dissertation\\agentic-rag-university-assistant\\data\\raw\\pdfs\\Academic-policy_Академическая политика 2025-2026 (англ).pdf'}

Content Preview (first 500 chars):
----------------------------------------
NON-PROFIT JOINT STOCK COMPANY 
«AL-FARABI KAZAKH NATIONAL UNIVERSITY» 
 
 
 
 
APPROVED 
by the decision of the Academic Council 
Protocol No. 11 of 27 June 2023 
(With amendments from August 1

Cleaning: 100%|██████████| 98/98 [00:00<00:00, 521.01it/s]


✓ Text cleaning completed
Splitting 98 documents into chunks...

Created 831 total chunks
Average chunk size: 718 characters

SAMPLE CHUNK:
Metadata: {'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-05T14:42:26+05:00', 'author': 'ˆ@C3>2KE ˘;LO', 'moddate': '2025-11-05T14:42:26+05:00', 'title': 'Microsoft Word - \x10:045< ?>;8B8:0 0=3; 2025 -2026 0=3; (1) final', 'source': 'Academic-policy_Академическая политика 2025-2026 (англ).pdf', 'total_pages': 33, 'page': 0, 'page_label': '1', 'source_type': 'pdf', 'file_path': 'd:\\agentic-rag-dissertation\\agentic-rag-university-assistant\\data\\raw\\pdfs\\Academic-policy_Академическая политика 2025-2026 (англ).pdf', 'chunk_id': 0, 'chunk_size': 243}

Content Preview (first 300 chars):
----------------------------------------
NON-PROFIT JOINT STOCK COMPANY 
AL-FARABI KAZAKH NATIONAL UNIVERSITY 

APPROVED 
by the decision of the Academic Council 
Protocol No. 11 of 27 June 2023 
(With amendments from August 1, 

## 📊 Summary

In this notebook, a complete data ingestion and preprocessing pipeline was successfully implemented for the KazNU RAG system.

### ✅ Key Achievements

- Loaded **PDF documents** and **web pages** as primary data sources  
- Applied **text cleaning and normalization** to improve data quality  
- Generated **overlapping text chunks** to preserve contextual meaning  
- Enriched each chunk with **metadata** (source, type, file path) for traceability  
- Performed **statistical analysis** to evaluate chunk distribution and sizes  
- Saved processed data in both:
  - **Pickle format** for system integration  
  - **JSON format** for inspection and validation  

### 📈 Output Overview

- Total Documents Processed: **{len(all_documents)}**
- Total Chunks Created: **{len(chunks)}**
- Average Chunk Size: **~{avg_chunk_size} characters**

### 🚀 Next Steps

The processed chunks generated in this notebook will be used in **Notebook 02** to:

- Generate semantic embeddings using a transformer-based model  
- Store embeddings in a **vector database (ChromaDB)**  
- Enable efficient semantic search for the RAG system  

This completes the **data preparation stage**, forming the backbone of the intelligent university information assistant.